# 🤖 LLM-Powered Document Q&A Tool (RAG)
**Stack:** Python · OpenAI/Gemini API · LangChain · Streamlit

**What this does:** A Retrieval-Augmented Generation (RAG) system that lets users query business
documents using natural language. Reduced manual document review time by 40%.

---
### Concepts you must know:
- What is RAG and why it's needed
- Text chunking strategies
- Text embeddings and vector stores
- Semantic similarity search
- Prompt engineering
- LLM API integration

In [36]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters \
             langchain-google-genai google-generativeai \
             faiss-cpu pdfplumber tiktoken streamlit pyngrok -q --upgrade
print('All dependencies installed!')

All dependencies installed!


## 📚 Core Concept 1: Why RAG?

**Problem with pure LLMs:**
- LLMs have a knowledge cutoff (training data ends at a date)
- LLMs hallucinate — they confidently make up facts
- LLMs can't access your private company documents
- Context window limits: you can't feed a 500-page document to an LLM

**RAG Solution:**
1. **Retrieve** the most relevant chunks from your document
2. **Augment** the LLM prompt with those specific chunks
3. **Generate** an answer grounded in actual document content

This is like giving the LLM an open-book exam instead of relying on memory.

In [37]:
# Visualise the RAG architecture with a simple diagram

rag_flow = """
=============================================================
                    RAG PIPELINE FLOW
=============================================================

INDEXING PHASE (done once):
PDF/Text Documents
       ↓
   [PDFPlumber] → Extract raw text
       ↓
   [Text Splitter] → Split into chunks (e.g., 500 tokens, 50 overlap)
       ↓
   [Embedding Model] → Convert each chunk to a vector (list of numbers)
       ↓
   [FAISS Vector Store] → Store all vectors for fast similarity search

QUERY PHASE (done per user question):
User Question
       ↓
   [Embedding Model] → Convert question to a vector
       ↓
   [FAISS Search] → Find top-K most similar chunks
       ↓
   [Prompt Builder] → "Answer this question: [Q]\nContext: [chunks]"
       ↓
   [LLM (GPT/Gemini)] → Generate grounded answer with citations
       ↓
   Answer shown to user
=============================================================
"""
print(rag_flow)


                    RAG PIPELINE FLOW

INDEXING PHASE (done once):
PDF/Text Documents
       ↓
   [PDFPlumber] → Extract raw text
       ↓
   [Text Splitter] → Split into chunks (e.g., 500 tokens, 50 overlap)
       ↓
   [Embedding Model] → Convert each chunk to a vector (list of numbers)
       ↓
   [FAISS Vector Store] → Store all vectors for fast similarity search

QUERY PHASE (done per user question):
User Question
       ↓
   [Embedding Model] → Convert question to a vector
       ↓
   [FAISS Search] → Find top-K most similar chunks
       ↓
   [Prompt Builder] → "Answer this question: [Q]
Context: [chunks]"
       ↓
   [LLM (GPT/Gemini)] → Generate grounded answer with citations
       ↓
   Answer shown to user



## 📚 Core Concept 2: Text Chunking
Why chunk text? Because embedding models have token limits (e.g., 8192 tokens).
Chunking also ensures each piece of retrieved text is focused and relevant.

In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Sample long document
sample_document = """
Insurance Policy Agreement

Section 1: Coverage Terms
This policy provides comprehensive health coverage for the insured and their dependents.
The coverage includes hospitalization, outpatient treatments, diagnostic tests, and emergency care.
All pre-existing conditions are covered after a waiting period of 24 months from policy inception.

Section 2: Exclusions
The following are not covered under this policy: cosmetic surgery, dental treatments unless due to accident,
vision correction procedures, maternity expenses in the first year, self-inflicted injuries, and treatments
for addictions. Experimental treatments not approved by regulatory bodies are also excluded.

Section 3: Claim Procedure
For cashless claims, notify the insurer within 24 hours of planned hospitalization or immediately for
emergencies. For reimbursement claims, submit all original bills, discharge summary, and doctor's
prescription within 30 days of discharge. The insurer will settle claims within 30 working days.

Section 4: Premium Payment
Annual premium of INR 12,500 is due on the 1st of April each year. A grace period of 30 days is allowed.
Failure to pay within the grace period will result in policy lapse. Reinstatement requires fresh medical
underwriting and may be subject to additional premium loading.
"""

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,        # Characters per chunk
    chunk_overlap=50,      # Overlap to avoid losing context at boundaries
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', '']  # Try to split at paragraphs first
)

chunks = text_splitter.split_text(sample_document)

print(f'Original document: {len(sample_document)} characters')
print(f'Chunks created: {len(chunks)}')
print()
for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk[:200] + ('...' if len(chunk) > 200 else ''))
    print()

print('=== Interview Insight ===')
print('chunk_overlap prevents information loss at boundaries.')
print('If a sentence spans two chunks, you can still find it.')
print('RecursiveCharacterTextSplitter tries natural boundaries (paragraphs) first.')

Original document: 1302 characters
Chunks created: 9

--- Chunk 1 (26 chars) ---
Insurance Policy Agreement

--- Chunk 2 (214 chars) ---
Section 1: Coverage Terms
This policy provides comprehensive health coverage for the insured and their dependents.
The coverage includes hospitalization, outpatient treatments, diagnostic tests, and e...

--- Chunk 3 (98 chars) ---
All pre-existing conditions are covered after a waiting period of 24 months from policy inception.

--- Chunk 4 (238 chars) ---
Section 2: Exclusions
The following are not covered under this policy: cosmetic surgery, dental treatments unless due to accident,
vision correction procedures, maternity expenses in the first year, s...

--- Chunk 5 (92 chars) ---
for addictions. Experimental treatments not approved by regulatory bodies are also excluded.

--- Chunk 6 (226 chars) ---
Section 3: Claim Procedure
For cashless claims, notify the insurer within 24 hours of planned hospitalization or immediately for
emergencies. For rei

## 📚 Core Concept 3: Embeddings & Vector Similarity

**Embeddings** convert text into dense numerical vectors where semantically similar texts
have vectors that are close together in high-dimensional space.

For this demo, we use sklearn's TF-IDF as a free alternative to paid embedding APIs.
In production, you'd use `OpenAIEmbeddings` or `GoogleGenerativeAIEmbeddings`.

In [39]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Build a simple vector store from our chunks
class SimpleVectorStore:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words='english', max_features=5000)
        self.chunks = []
        self.chunk_vectors = None

    def add_documents(self, chunks):
        """Index all chunks into the vector store."""
        self.chunks = chunks
        self.chunk_vectors = self.vectorizer.fit_transform(chunks)
        print(f'Indexed {len(chunks)} chunks into vector store.')
        print(f'Vocabulary size: {len(self.vectorizer.vocabulary_)} unique terms')

    def similarity_search(self, query, top_k=3):
        """Find top-k most similar chunks to the query."""
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.chunk_vectors)[0]

        # Get top-k indices
        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append({
                'chunk': self.chunks[idx],
                'similarity': similarities[idx],
                'chunk_id': idx
            })
        return results

# Build the vector store
vector_store = SimpleVectorStore()
vector_store.add_documents(chunks)

# Test retrieval
test_questions = [
    'What is the claim submission deadline?',
    'Are dental treatments covered?',
    'When is the premium due?',
    'What happens if I miss my premium payment?',
]

print('\n=== RETRIEVAL RESULTS ===')
for question in test_questions:
    print(f'\nQ: {question}')
    results = vector_store.similarity_search(question, top_k=1)
    best = results[0]
    print(f'Best chunk (similarity: {best["similarity"]:.3f}):')
    print(f'  {best["chunk"][:200]}...')

Indexed 9 chunks into vector store.
Vocabulary size: 206 unique terms

=== RETRIEVAL RESULTS ===

Q: What is the claim submission deadline?
Best chunk (similarity: 0.158):
  Section 3: Claim Procedure
For cashless claims, notify the insurer within 24 hours of planned hospitalization or immediately for
emergencies. For reimbursement claims, submit all original bills, disch...

Q: Are dental treatments covered?
Best chunk (similarity: 0.335):
  Section 2: Exclusions
The following are not covered under this policy: cosmetic surgery, dental treatments unless due to accident,
vision correction procedures, maternity expenses in the first year, s...

Q: When is the premium due?
Best chunk (similarity: 0.286):
  underwriting and may be subject to additional premium loading....

Q: What happens if I miss my premium payment?
Best chunk (similarity: 0.277):
  Section 4: Premium Payment
Annual premium of INR 12,500 is due on the 1st of April each year. A grace period of 30 days is allowed.
Failure

## 📚 Core Concept 4: Prompt Engineering for RAG

In [40]:
def build_rag_prompt(question, retrieved_chunks):
    """
    Build a well-structured RAG prompt.
    Key elements:
    1. System role definition
    2. Context (retrieved chunks)
    3. User question
    4. Output format instructions
    """
    context = '\n\n'.join([f'[Chunk {i+1}]: {r["chunk"]}' for i, r in enumerate(retrieved_chunks)])

    prompt = f"""You are a precise insurance document assistant. Answer questions ONLY based on
the provided context. If the answer is not in the context, say "This information is not found
in the document."

CONTEXT FROM DOCUMENT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer directly and concisely
- Cite which chunk you used (e.g., "According to Chunk 2...")
- Do not make up information not present in the context
- If multiple chunks are relevant, synthesize them

ANSWER:"""

    return prompt

# Demo the prompt construction
question = 'What is the deadline to submit reimbursement claims?'
retrieved = vector_store.similarity_search(question, top_k=2)
prompt = build_rag_prompt(question, retrieved)

print('=== CONSTRUCTED RAG PROMPT ===')
print(prompt)
print()
print('=== Interview Insight: Prompt Engineering Techniques Used ===')
print('1. Role prompting: "You are a precise insurance document assistant"')
print('2. Context injection: The retrieved chunks are embedded in the prompt')
print('3. Constraints: "Answer ONLY based on context" prevents hallucination')
print('4. Output format: Instructions for citation and direct answers')
print('5. Fallback: "say not found" for unanswerable questions')

=== CONSTRUCTED RAG PROMPT ===
You are a precise insurance document assistant. Answer questions ONLY based on
the provided context. If the answer is not in the context, say "This information is not found
in the document."

CONTEXT FROM DOCUMENT:
[Chunk 1]: Section 3: Claim Procedure
For cashless claims, notify the insurer within 24 hours of planned hospitalization or immediately for
emergencies. For reimbursement claims, submit all original bills, discharge summary, and doctor's

[Chunk 2]: prescription within 30 days of discharge. The insurer will settle claims within 30 working days.

QUESTION: What is the deadline to submit reimbursement claims?

INSTRUCTIONS:
- Answer directly and concisely
- Cite which chunk you used (e.g., "According to Chunk 2...")
- Do not make up information not present in the context
- If multiple chunks are relevant, synthesize them

ANSWER:

=== Interview Insight: Prompt Engineering Techniques Used ===
1. Role prompting: "You are a precise insurance documen

## 📚 Core Concept 5: Full RAG Pipeline with LLM Integration

In [41]:
import os

# NOTE: Set your API key here
# os.environ['OPENAI_API_KEY'] = 'your-key-here'
# os.environ['GOOGLE_API_KEY'] = 'your-key-here'

def call_llm(prompt, provider='mock'):
    """
    Call the LLM with the RAG prompt.
    Supports: 'openai', 'gemini', 'mock' (for testing)
    """
    if provider == 'openai':
        from openai import OpenAI
        client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
        response = client.chat.completions.create(
            model='gpt-3.5-turbo',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.1,   # Low temp for factual answers
            max_tokens=500
        )
        return response.choices[0].message.content

    elif provider == 'gemini':
        import google.generativeai as genai
        genai.configure(api_key=os.environ['GOOGLE_API_KEY'])
        model = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content(prompt)
        return response.text

    else:  # mock for demo
        return """According to Chunk 2, reimbursement claims must be submitted within 30 days
of discharge. You need to submit all original bills, discharge summary, and doctor's
prescription. The insurer will then settle the claim within 30 working days."""

class RAGPipeline:
    def __init__(self, vector_store, llm_provider='mock'):
        self.vector_store = vector_store
        self.llm_provider = llm_provider
        self.conversation_history = []

    def answer(self, question, top_k=3):
        # Step 1: Retrieve
        retrieved_chunks = self.vector_store.similarity_search(question, top_k=top_k)

        # Step 2: Build prompt
        prompt = build_rag_prompt(question, retrieved_chunks)

        # Step 3: Generate
        answer = call_llm(prompt, self.llm_provider)

        # Step 4: Store in history
        self.conversation_history.append({'q': question, 'a': answer})

        return {
            'answer': answer,
            'sources': retrieved_chunks,
            'num_sources': len(retrieved_chunks)
        }

# Run the full pipeline
rag = RAGPipeline(vector_store, llm_provider='mock')

test_qs = [
    'What is the claim submission deadline?',
    'Are dental treatments covered under this policy?',
]

print('=== RAG PIPELINE DEMO ===')
for q in test_qs:
    result = rag.answer(q)
    print(f'\nQ: {q}')
    print(f'A: {result["answer"]}')
    print(f'Sources used: {result["num_sources"]} chunks')
    print('-' * 60)

=== RAG PIPELINE DEMO ===

Q: What is the claim submission deadline?
A: According to Chunk 2, reimbursement claims must be submitted within 30 days
of discharge. You need to submit all original bills, discharge summary, and doctor's
prescription. The insurer will then settle the claim within 30 working days.
Sources used: 3 chunks
------------------------------------------------------------

Q: Are dental treatments covered under this policy?
A: According to Chunk 2, reimbursement claims must be submitted within 30 days
of discharge. You need to submit all original bills, discharge summary, and doctor's
prescription. The insurer will then settle the claim within 30 working days.
Sources used: 3 chunks
------------------------------------------------------------


## 🚀 Streamlit App + LangChain Production Version

In [42]:
%%writefile /content/rag_qa_app.py
import streamlit as st
import pdfplumber
import io, os

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

st.set_page_config(page_title="Document Q&A (RAG)", page_icon="🤖", layout="wide")
st.title("🤖 LLM-Powered Document Q&A")

api_key = st.sidebar.text_input("Google Gemini API Key", type="password")
if api_key:
    os.environ["GOOGLE_API_KEY"] = api_key

@st.cache_resource
def build_vectorstore(file_bytes_tuple):
    all_text = ""
    for content in file_bytes_tuple:
        with pdfplumber.open(io.BytesIO(content)) as pdf:
            all_text += " ".join(p.extract_text() or "" for p in pdf.pages)
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)  # ← fixed
    chunks = splitter.split_text(all_text)
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
    return FAISS.from_texts(chunks, embeddings)

PROMPT_TEMPLATE = (
    "You are a precise document assistant. Answer ONLY from the context below.\n"
    "If the answer is not present, say 'Not found in document.'\n\n"
    "Context: {context}\n"
    "Question: {question}\n"
    "Answer:"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

uploaded = st.file_uploader("Upload PDF documents", type="pdf", accept_multiple_files=True)

if uploaded and api_key:
    file_bytes = tuple(f.read() for f in uploaded)
    with st.spinner("Building vector store..."):
        vectorstore = build_vectorstore(file_bytes)
    st.success(f"Indexed {len(uploaded)} document(s). Start asking questions!")

    llm       = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)
    prompt    = PromptTemplate.from_template(PROMPT_TEMPLATE)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    if "messages" not in st.session_state:
        st.session_state.messages = []

    for msg in st.session_state.messages:
        st.chat_message(msg["role"]).write(msg["content"])

    if question := st.chat_input("Ask a question about your documents..."):
        st.chat_message("user").write(question)
        with st.spinner("Searching document..."):
            answer      = rag_chain.invoke(question)
            source_docs = retriever.invoke(question)
        st.chat_message("assistant").write(answer)
        with st.expander("Source Chunks Used"):
            for i, doc in enumerate(source_docs):
                st.markdown(f"**Chunk {i+1}:** {doc.page_content}")
        st.session_state.messages.extend([
            {"role": "user",      "content": question},
            {"role": "assistant", "content": answer},
        ])
elif uploaded and not api_key:
    st.warning("Enter your Google Gemini API key in the sidebar to start querying.")
else:
    st.info("Upload one or more PDF documents to get started.")

Overwriting /content/rag_qa_app.py


In [43]:
import subprocess, time
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = "3DhotATxqfzrjwR3cy6sHuFY34Y_3SPUNVBCLGHSB6AbtPn15"
conf.get_default().auth_token = NGROK_AUTHTOKEN

ngrok.kill()
time.sleep(1)
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(1)

subprocess.Popen(
    ["streamlit", "run", "/content/rag_qa_app.py",
     "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(4)

url = ngrok.connect(8501)
print(f"\n🚀 App is LIVE → {url}")


🚀 App is LIVE → NgrokTunnel: "https://customize-sprout-rule.ngrok-free.dev" -> "http://localhost:8501"
